In [1]:
# Load model
import torch
from transformers import pipeline

model_id = "meta-llama/Llama-3.2-3B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"done! model {model_id} loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

done! modelo meta-llama/Llama-3.2-3B-Instruct carregado.


In [2]:
# Extract json area
import json
import os

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(var,filename):
    var = clean_out(var)
    with open(out_json(filename), "w", encoding="utf-8") as f:
        json.dump(var,f,ensure_ascii=False, indent=4)

def clean_out(var):
    var = var.replace('```json','').replace('```','').strip()
    var_convert=json.loads(var)
    return  var_convert
    
# Feed with your own evidence in the indicate path
out_dir = './maidai/evidence'

def out_json(filename):
    return f'{out_dir}/{filename}.json'

audio=f'{out_dir}/audio.json'
text_message=f'{out_dir}/text_message.json'
image=f'{out_dir}/image.json'

audio_data = read_json(audio)
image_data = read_json(image)
text_message_data = read_json(text_message)

In [11]:
# Prompt for entite extract
entities = f""" Analyze the following digital evidence and identify the relevant entities that appear in it. 

EVIDENCE SOURCES: 
AUDIO: {json.dumps(audio_data, ensure_ascii=False, indent=2)} 
IMAGE: {json.dumps(image_data, ensure_ascii=False, indent=2)} 
WHATSAPP: {json.dumps(text_message_data, ensure_ascii=False, indent=2)} 

OUTPUT INSTRUCTIONS: 
Return ONLY a valid JSON object matching this exact schema. Do NOT add conversational text before or after the JSON. 

Required Output Schema: 
{{ 
  "entities": [ 
    {{ 
      "entity_id": "ENT_001", 
      "name_or_alias": "Explicit name, nickname, alias, phone number, vehicle plate, or label", 
      "entity_type": "person / organization / location / object / quantity / date_time / vehicle / financial_info", 
      "evidence_source": "Exact source (AUDIO, IMAGE, or WHATSAPP)", 
      "location_within_source": "Audio segment, WhatsApp message index, OCR field, or image ID" 
    }} 
  ] 
}} 
""" 

messages_entities = [ 
    { 
        "role": "system", 
        "content": ( 
            "You are a digital evidence analysis model dedicated solely to Stage 1 (Entity Extraction). " 
            "Focus on identifying people, names, nicknames, organizations, locations, objects, vehicles, and financial info. " 
            "Identify entities strictly from explicit evidence. Do not assume a person in an image matches a nickname from a chat. " 
            "Do not extract events or relationships in this stage. " 
            "Return ONLY a valid JSON following the exact schema requested." 
        ) 
    }, 
    { 
        "role": "user", 
        "content": entities 
    } 
] 

outputs_entities = pipe( 
    messages_entities, 
    max_new_tokens=4096, 
    do_sample=False 
) 

entities_result = outputs_entities[0]["generated_text"][-1]["content"]

print(clean_out(entities_result))

write_json(entities_result, "entities")

[transformers] Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'entities': [{'entity_id': 'ENT_001', 'name_or_alias': 'Gordo', 'entity_type': 'person', 'evidence_source': 'AUDIO', 'location_within_source': 'segments[0].text'}, {'entity_id': 'ENT_002', 'name_or_alias': 'Zé', 'entity_type': 'person', 'evidence_source': 'AUDIO', 'location_within_source': 'segments[1].text'}, {'entity_id': 'ENT_003', 'name_or_alias': 'Menor', 'entity_type': 'person', 'evidence_source': 'AUDIO', 'location_within_source': 'segments[2].text'}, {'entity_id': 'ENT_004', 'name_or_alias': 'Rua das Palmeiras, 127, Aracaju-SE', 'entity_type': 'location', 'evidence_source': 'IMAGE', 'location_within_source': 'metadata.captured_at'}, {'entity_id': 'ENT_005', 'name_or_alias': 'Banco Exemplo', 'entity_type': 'organization', 'evidence_source': 'WHATSAPP', 'location_within_source': 'messages[2].text'}, {'entity_id': 'ENT_006', 'name_or_alias': 'Pix', 'entity_type': 'financial_info', 'evidence_source': 'IMAGE', 'location_within_source': 'analysis.description'}, {'entity_id': 'ENT_00

In [10]:
# Prompt for event extract

events = f"""Você é um modelo de análise forense de evidências.

SEPARAÇÃO OBRIGATÓRIA DE FONTES:

Ao extrair eventos do ÁUDIO, utilize exclusivamente a transcrição e os metadados do áudio. 
Ao extrair eventos de IMAGEM, utilize exclusivamente a descrição visual, o OCR e os metadados da imagem. 
Nunca transfira uma informação de uma fonte para outra.

EXEMPLO CORRETO:

ÁUDIO seg_002: "É pra repassar pro Menor"
→ EVT_XX: descricao="É pra repassar pro Menor", fonte=AUDIO, local_na_fonte=seg_002

IMAGEM IMG_0342: "Um homem segurando pistola"
→ EVT_YY: descricao="Um homem está segurando uma pistola", fonte=IMAGEM, local_na_fonte=IMG_0342

NÃO FAÇA:
ÁUDIO marcado como IMAGEM
Descrição do ÁUDIO em evento IMAGEM
Ignorar IMAGEM porque ÁUDIO fala de outra coisa

PROCESSE TODAS AS 3 FONTES INDEPENDENTEMENTE

EVIDÊNCIAS:

ÁUDIO:
{json.dumps(audio_data, ensure_ascii=False, indent=2)}

IMAGEM:
{json.dumps(image_data, ensure_ascii=False, indent=2)}

WHATSAPP:
{json.dumps(text_message_data, ensure_ascii=False, indent=2)}

REGRAS:

1. Examine cada fonte separadamente. Inclua todos os eventos encontrados, mesmo se similar em outra fonte.

2. Extraia apenas o que está explicitamente na evidência. Diferencie:
   - Ação planejada (alguém diz que vai fazer)
   - Ação comunicada (alguém fala sobre uma ação)
   - Ação observada (visível na imagem/áudio)

3. Timestamp do evento = quando a ação ocorre/ocorreu (pode ser null ou relativo).
   Timestamp da evidência = quando o arquivo foi gerado.
   Não confunda.

4. Inclua apenas nomes/apelidos explicitamente mencionados.
   Não resolva se dois nomes são a mesma pessoa.
   Não atribua ações a pessoas não mencionadas na fonte.

5. event_type: use um de: "comunicacao", "transacao", "encontro", "entrega", "negociacao", "acao"

6. evidence_source: "AUDIO", "IMAGEM" ou "WHATSAPP"
   location_within_source: ID da mensagem, segmento ou evidência

7. Valide antes de retornar: todas as 3 fontes examinadas, cada evento citado, timestamps corretos.

RETORNE APENAS JSON VÁLIDO:

{{
  "eventos": [
    {{
      "event_id": "EVT_001",
      "tipo": "comunicacao",
      "descricao": "descrição suportada pela evidência",
      "pessoas_mencionadas": ["nome"],
      "timestamp_evento": null,
      "localizacao_evento": null,
      "fonte": "WHATSAPP",
      "timestamp_evidencia": null,
      "local_na_fonte": "msg_001"
    }}
  ]
}}
"""

messages_events = [
    {
        "role": "system",
        "content": (
            """Você é um modelo de análise de evidências digitais. Sua tarefa é extrair eventos 
            das fontes fornecidas com rigor forense.
            
            INSTRUÇÃO CRÍTICA:
            Faça todo seu raciocínio internamente. Na saída, retorne APENAS o JSON válido.
            Nada de preamble, explicações, observações ou texto fora do JSON.
            Somente o JSON.

            PRINCÍPIOS:
            - Baseie-se exclusivamente nas evidências fornecidas
            - Não invente fatos, ações, participantes ou localizações
            - Não resolva identidades: se dois nomes podem ser a mesma pessoa, deixe separado
            - Distingua: ação planejada (será feita) vs. ação concluída (foi feita)
            - Não confunda: timestamp do evento (quando ocorreu) com timestamp da evidência (quando foi registrada)
            
            PROCEDIMENTO:
            1. Examine cada fonte (ÁUDIO, IMAGEM, WHATSAPP) independentemente
            2. Extraia apenas eventos explicitamente suportados
            3. Cite a fonte e a localização exata em cada evento
            4. Retorne APENAS JSON válido no esquema especificado
            """
        )
    },
    {
        "role": "user",
        "content": events
    }
]

outputs_events = pipe(
    messages_events,
    max_new_tokens=2048,
    do_sample=False
)

events_result = outputs_events[0]["generated_text"][-1]["content"]
print(clean_out(events_result))
write_json(events_result, "events")

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'eventos': [{'event_id': 'EVT_001', 'tipo': 'comunicacao', 'descricao': 'Zé, conseguiu separar os pinos?', 'pessoas_mencionadas': ['Gordo', 'Zé'], 'timestamp_evento': '2026-10-06T17:42:18-03:00', 'localizacao_evento': None, 'fonte': 'WHATSAPP', 'timestamp_evidencia': None, 'local_na_fonte': 'msg_001'}, {'event_id': 'EVT_002', 'tipo': 'comunicacao', 'descricao': 'Separei. Vou deixar na casa da Rua das Palmeiras.', 'pessoas_mencionadas': ['Gordo', 'Zé'], 'timestamp_evento': '2026-10-06T17:43:02-03:00', 'localizacao_evento': None, 'fonte': 'WHATSAPP', 'timestamp_evidencia': None, 'local_na_fonte': 'msg_002'}, {'event_id': 'EVT_003', 'tipo': 'comunicacao', 'descricao': 'Fechado. O Menor pega depois das oito.', 'pessoas_mencionadas': ['Gordo', 'Zé'], 'timestamp_evento': '2026-10-06T17:44:31-03:00', 'localizacao_evento': None, 'fonte': 'WHATSAPP', 'timestamp_evidencia': None, 'local_na_fonte': 'msg_003'}, {'event_id': 'EVT_004', 'tipo': 'entrega', 'descricao': 'É pra repassar pro Menor que 

In [44]:
#PEGA SAIDAS JSON 
entidades=f'{out_dir}/entities.json'
eventos=f'{out_dir}/events.json'

entities_data = read_json(entidades)
events_data = read_json(eventos)

In [45]:
# Prompt for relate outputs
relate_outputs = f""" Analyze the provided ENTITIES and EVENTS to establish explicit Entity-Event relationships. 
ENTITIES: {json.dumps(entities_data, ensure_ascii=False, indent=2)} 
EVENTS: {json.dumps(events_data, ensure_ascii=False, indent=2)} 

OUTPUT INSTRUCTIONS: 
Return ONLY a valid JSON object matching this exact schema. Do NOT repeat the full entities or events lists in your response. 

Required Output Schema: 
{{ 
  "entity_event_relationships": [ 
    {{ 
      "entity": "Name or ID of the entity", 
      "event": "Short description or ID of the event", 
      "role": "Role played (e.g., sender, participant, receiver, owner, location)", 
      "evidence_source": "Exact evidence file or reference", 
      "location_within_source": "Timestamp, message index, OCR line, or audio segment" 
    }} 
  ] 
}} 
""" 

messages_relate = [ 
    { 
        "role": "system", 
        "content": ( 
            "You are a digital evidence analysis model focused solely on Stage 2 (Entity-Event extraction). " 
            "Your task is to link identified entities directly to their respective events based EXCLUSIVELY " 
            "on explicit evidence. Do not invent identities, links, or assumptions. " 
            "Your output must contain ONLY the merged relationships JSON schema requested by the user." 
        ) 
    }, 
    { 
        "role": "user", 
        "content": relate_outputs 
    } 
] 

outputs_relate = pipe( 
    messages_relate, 
    max_new_tokens=4096, 
    do_sample=False # Desativa o amostragem para evitar alucinações de formato 
) 

relate_result = outputs_merge[0]["generated_text"][-1]["content"]
print(merge_result)

[transformers] Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "entity_event_relationships": [
    {
      "entity": "Gordo",
      "event": "EVT_001",
      "role": "sender",
      "evidence_source": "WHATSAPP",
      "location_within_source": "msg_001"
    },
    {
      "entity": "Zé",
      "event": "EVT_001",
      "role": "receiver",
      "evidence_source": "WHATSAPP",
      "location_within_source": "msg_001"
    },
    {
      "entity": "Gordo",
      "event": "EVT_003",
      "role": "sender",
      "evidence_source": "WHATSAPP",
      "location_within_source": "msg_003"
    },
    {
      "entity": "Gordo",
      "event": "EVT_002",
      "role": "participant",
      "evidence_source": "WHATSAPP",
      "location_within_source": "msg_002"
    },
    {
      "entity": "Zé",
      "event": "EVT_002",
      "role": "sender",
      "evidence_source": "WHATSAPP",
      "location_within_source": "msg_002"
    },
    {
      "entity": "Zé",
      "event": "EVT_004",
      "role": "sender",
      "evidence_source": "IMAGE",
      "l